In [ ]:
import os
import re
import glob
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

# ============================================================
# INDEPENDENT IMAGE EVALUATION - CLASSIFICATION ONLY
# - Each image is evaluated independently
# - Does NOT use sequences, HOLD/FIRE, or temporal matching
# - Matches by filename using cls_labels
# - Exports an Excel file with per-image predictions and the confusion matrix
# ============================================================

# ======================= EDIT ================================
os.chdir('/mmsegmentation')

CONFIG = '/mmsegmentation/zmax_configs/for_test_hasta_26_3/bisenet_cls_only_test_clean.py'
CKPT_OR_WORKDIR = '/mmsegmentation/work_dirs/bisenet_cls_only_strict_fixed/best_cls_acc_cls_top1_iter_36400.pth'

DEVICE = 'cuda:0'

IMG_DIR      = '/automine1d_cls/img_dir/val_aug/'
CLS_ANN_PATH = '/automine1d_cls/cls_labels_val_aug_fixed.txt'
OUTPUT_XLSX  = '/mmsegmentation/output/eval_bisenet_cls_only_val.xlsx'

IMG_EXTS = {'.png', '.jpg', '.jpeg', '.bmp'}

# MODEL class names (classifier order)
MODEL_CLS_NAMES = ['izquierda', 'recta', 'derecha']

# Mapping from GT labels in the txt file to model class names
GT_LABEL_MAP = {
    'LEFT': 'izquierda',
    'STRAIGHT': 'recta',
    'RIGHT': 'derecha',
    'IZQUIERDA': 'izquierda',
    'RECTA': 'recta',
    'DERECHA': 'derecha',
}

# Inference consistent with the video script / previous notebook
PREPROCESS_USE_PINNED = True
USE_AUTOCAST = True
# ============================================================

import os.path as osp
from mmseg.apis import init_model
from mmseg.utils import register_all_modules

try:
    from mmengine.structures import LabelData
except Exception:
    LabelData = None

try:
    from mmseg.structures.dual_task_seg_data_sample import DualTaskSegDataSample as _TestDataSample
except Exception:
    from mmseg.structures import SegDataSample as _TestDataSample


def ensure_dir_for_file(path: str):
    Path(path).parent.mkdir(parents=True, exist_ok=True)


def parse_timestamp_from_name(path: str) -> Optional[float]:
    """Kept only as auxiliary metadata; it is NOT used for evaluation."""
    stem = Path(path).stem
    try:
        return float(stem)
    except Exception:
        pass
    matches = re.findall(r'(?<!\d)(\d{10}(?:\.\d+)?)(?!\d)', stem)
    if matches:
        try:
            return float(matches[-1])
        except Exception:
            return None
    return None


def sorted_image_paths(img_dir: str) -> List[str]:
    paths = [str(p) for p in Path(img_dir).iterdir() if p.suffix.lower() in IMG_EXTS]
    return sorted(paths, key=lambda p: Path(p).name)


def normalize_name_key(name: str) -> Tuple[str, str]:
    base = os.path.basename(name)
    stem = os.path.splitext(base)[0]
    return base, stem


def normalize_gt_label(raw_label: str) -> str:
    key = str(raw_label).strip().upper()
    if key in GT_LABEL_MAP:
        return GT_LABEL_MAP[key]
    raise KeyError(f'La etiqueta GT "{raw_label}" no existe en GT_LABEL_MAP.')


def parse_cls_annotations(path: str):
    ann_by_name = {}
    ann_list = []

    with open(path, 'r', encoding='utf-8') as f:
        for raw in f:
            line = raw.strip()
            if not line or line.startswith('#'):
                continue

            parts = re.split(r'[\s,;]+', line)
            if len(parts) < 2:
                continue

            name_tok = parts[0]
            label_tok = parts[1]
            base, stem = normalize_name_key(name_tok)
            ts = parse_timestamp_from_name(name_tok)
            model_label = normalize_gt_label(label_tok)
            model_idx = MODEL_CLS_NAMES.index(model_label)

            info = {
                'raw_label': label_tok,
                'gt_cls_name': model_label,
                'gt_cls_idx': int(model_idx),
                'image_name': base,
                'image_stem': stem,
                'timestamp_s': ts,
            }
            ann_by_name[base] = info
            ann_by_name[stem] = info
            ann_list.append(info)

    return ann_by_name, ann_list


def safe_nanmean(series):
    vals = pd.to_numeric(series, errors='coerce').to_numpy(dtype=float)
    return float(np.nanmean(vals)) if np.any(~np.isnan(vals)) else np.nan


def cls_top1_from_cm(cm: np.ndarray) -> float:
    total = cm.sum()
    if total <= 0:
        return float('nan')
    return float(np.trace(cm) / total)


def add_cls_pair(cm: np.ndarray, gt_idx: int, pred_idx: int) -> np.ndarray:
    if 0 <= gt_idx < cm.shape[0] and 0 <= pred_idx < cm.shape[1]:
        cm[gt_idx, pred_idx] += 1
    return cm


def _label_from_LabelData(x):
    """Extracts an integer from different possible pred_label representations."""
    import numpy as _np
    import torch as _torch

    for _ in range(10):
        if x is None:
            return None
        if isinstance(x, _torch.Tensor):
            return int(x.reshape(-1)[0].detach().cpu().item())
        if isinstance(x, _np.ndarray):
            return int(x.reshape(-1)[0])
        if isinstance(x, (int, float, bool, _np.integer, _np.floating)):
            return int(x)
        if isinstance(x, (list, tuple)):
            if len(x) == 0:
                return None
            x = x[0]
            continue
        if hasattr(x, 'item') and callable(getattr(x, 'item')):
            try:
                return int(x.item())
            except Exception:
                pass
        next_x = None
        for k in ('label', 'pred_label', 'data', 'value'):
            if hasattr(x, k):
                next_x = getattr(x, k)
                break
        if next_x is x:
            break
        x = next_x
    return None


def resolve_checkpoint(path_or_dir: str) -> str:
    p = Path(path_or_dir)
    if p.is_file():
        return str(p)

    if not p.exists():
        raise FileNotFoundError(f'No existe CKPT_OR_WORKDIR: {path_or_dir}')

    patterns = [
        'best_cls_acc_cls_top1*.pth',
        'latest.pth',
        '*.pth',
    ]
    for pat in patterns:
        found = sorted(p.glob(pat))
        if found:
            # prefer the most recent one by mtime
            found = sorted(found, key=lambda x: x.stat().st_mtime, reverse=True)
            return str(found[0])

    raise FileNotFoundError(f'No se encontro ningun checkpoint util en: {path_or_dir}')


class NDArrayClsInferencer:
    def __init__(self, model):
        self.model = model
        self.model.eval()
        self.device = next(self.model.parameters()).device

        cfg_dp = model.cfg.model.get('data_preprocessor', {})
        size = cfg_dp.get('size', (512, 512))
        self.input_w = int(size[0])
        self.input_h = int(size[1])
        self.bgr_to_rgb = bool(cfg_dp.get('bgr_to_rgb', True))
        mean = np.array(cfg_dp.get('mean', [123.675, 116.28, 103.53]), dtype=np.float32)
        std = np.array(cfg_dp.get('std', [58.395, 57.12, 57.375]), dtype=np.float32)

        self.mean = torch.tensor(mean, device=self.device, dtype=torch.float32).view(1, 3, 1, 1)
        self.std = torch.tensor(std, device=self.device, dtype=torch.float32).view(1, 3, 1, 1)

        self._meta = dict(
            ori_shape=(self.input_h, self.input_w),
            img_shape=(self.input_h, self.input_w),
            pad_shape=(self.input_h, self.input_w),
            batch_input_shape=(self.input_h, self.input_w),
            scale_factor=(1.0, 1.0),
            padding_size=[0, 0, 0, 0],
            flip=False,
            flip_direction=None,
        )

        self.use_cuda = (self.device.type == 'cuda')
        self.use_pinned = bool(PREPROCESS_USE_PINNED and self.use_cuda)

        self._resize_hwc = np.empty((self.input_h, self.input_w, 3), dtype=np.uint8)
        self._cpu_hwc_t = None
        self._cpu_hwc_np = None
        self._gpu_hwc_u8 = None
        self._gpu_chw_f32 = None

        if self.use_pinned:
            self._cpu_hwc_t = torch.empty((1, self.input_h, self.input_w, 3), dtype=torch.uint8, pin_memory=True)
            self._cpu_hwc_np = self._cpu_hwc_t[0].numpy()
            self._gpu_hwc_u8 = torch.empty(
                (1, self.input_h, self.input_w, 3),
                dtype=torch.uint8,
                device=self.device
            )
            self._gpu_chw_f32 = torch.empty(
                (1, 3, self.input_h, self.input_w),
                dtype=torch.float32,
                device=self.device
            )

    def _make_data_sample(self):
        ds = _TestDataSample()
        ds.set_metainfo(self._meta)
        return ds

    def _preprocess(self, img_bgr_nd):
        if img_bgr_nd.ndim != 3 or img_bgr_nd.shape[2] != 3:
            raise ValueError(f'Se esperaba HxWx3, llego {img_bgr_nd.shape}')

        cv2.resize(
            img_bgr_nd,
            (self.input_w, self.input_h),
            dst=self._resize_hwc,
            interpolation=cv2.INTER_LINEAR
        )

        if self.bgr_to_rgb:
            img = self._resize_hwc[..., ::-1]
        else:
            img = self._resize_hwc

        if self.use_pinned:
            np.copyto(self._cpu_hwc_np, img[None, ...])
            self._gpu_hwc_u8.copy_(self._cpu_hwc_t, non_blocking=True)
            x = self._gpu_hwc_u8.permute(0, 3, 1, 2)
            self._gpu_chw_f32.copy_(x, non_blocking=True)
            tensor = self._gpu_chw_f32
        else:
            img = np.ascontiguousarray(img.transpose(2, 0, 1))
            tensor = torch.from_numpy(img).unsqueeze(0)
            if self.use_cuda:
                tensor = tensor.to(self.device, non_blocking=True)
            tensor = tensor.float()

        tensor = (tensor - self.mean) / self.std
        return tensor

    @torch.inference_mode()
    def __call__(self, img_bgr_nd):
        inputs = self._preprocess(img_bgr_nd)
        data_samples = [self._make_data_sample()]

        if USE_AUTOCAST and self.use_cuda:
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                preds = self.model.predict(inputs, data_samples)
        else:
            preds = self.model.predict(inputs, data_samples)

        sample = preds[0]

        cls_idx = None
        if hasattr(sample, 'pred_label') and sample.pred_label is not None:
            cls_idx = _label_from_LabelData(sample.pred_label)

        if cls_idx is None:
            raise RuntimeError('No se encontro pred_label en el DataSample de salida.')

        pred_cls_name = MODEL_CLS_NAMES[int(cls_idx)] if 0 <= int(cls_idx) < len(MODEL_CLS_NAMES) else str(cls_idx)
        return int(cls_idx), pred_cls_name


def prepare_model():
    register_all_modules(init_default_scope=False)
    ckpt = resolve_checkpoint(CKPT_OR_WORKDIR)
    model = init_model(CONFIG, ckpt, device=DEVICE)
    infer = NDArrayClsInferencer(model)
    return model, infer, ckpt


def run_eval():
    img_paths = sorted_image_paths(IMG_DIR)
    if len(img_paths) == 0:
        raise RuntimeError(f'No se encontraron imagenes en {IMG_DIR}')

    ann_by_name, ann_list = parse_cls_annotations(CLS_ANN_PATH)
    _, infer, resolved_ckpt = prepare_model()

    results = []
    cls_cm = np.zeros((len(MODEL_CLS_NAMES), len(MODEL_CLS_NAMES)), dtype=np.int64)

    missing_cls_ann = 0

    for idx, img_path in enumerate(tqdm(img_paths, desc='Evaluando clasificacion independiente')):
        img_name = os.path.basename(img_path)
        img_stem = Path(img_name).stem
        timestamp = parse_timestamp_from_name(img_path)

        img_bgr = cv2.imread(img_path, cv2.IMREAD_COLOR)
        if img_bgr is None:
            raise FileNotFoundError(f'No se pudo leer imagen: {img_path}')

        pred_cls_idx, pred_cls_name = infer(img_bgr)

        gt_info = ann_by_name.get(img_name)
        if gt_info is None:
            gt_info = ann_by_name.get(img_stem)

        evaluated = bool(gt_info is not None)

        row = {
            'frame_idx': int(idx),
            'image_name': img_name,
            'timestamp_s': float(timestamp) if timestamp is not None else np.nan,
            'time_rel_s': float(idx),  # index only; it has no temporal meaning
            'evaluated_with_gt': int(evaluated),
            'pred_cls_idx': int(pred_cls_idx),
            'pred_cls_name': pred_cls_name,
            'gt_cls_idx': np.nan,
            'gt_cls_name': None,
            'gt_label_raw': None,
            'cls_correct': np.nan,
        }

        if gt_info is None:
            missing_cls_ann += 1
        else:
            gt_cls_idx = int(gt_info['gt_cls_idx'])
            gt_cls_name = str(gt_info['gt_cls_name'])
            cls_cm = add_cls_pair(cls_cm, gt_cls_idx, pred_cls_idx)

            row.update({
                'gt_cls_idx': gt_cls_idx,
                'gt_cls_name': gt_cls_name,
                'gt_label_raw': gt_info['raw_label'],
                'cls_correct': int(gt_cls_idx == int(pred_cls_idx)),
            })

        results.append(row)

    df = pd.DataFrame(results)
    eval_df = df[df['evaluated_with_gt'] == 1].copy()

    summary = {
        'evaluation_mode': 'independent_images_cls_only',
        'n_all_frames': int(len(df)),
        'n_annotated_images_in_txt': int(len(ann_list)),
        'n_evaluated_frames': int(len(eval_df)),
        'images_without_cls_annotation': int(missing_cls_ann),
        'cls_top1_eval_only': float(cls_top1_from_cm(cls_cm)),
        'cls_correct_count_eval_only': int(pd.to_numeric(eval_df['cls_correct'], errors='coerce').fillna(0).sum()) if len(eval_df) else 0,
        'config': CONFIG,
        'checkpoint': resolved_ckpt,
        'img_dir': IMG_DIR,
        'cls_ann_path': CLS_ANN_PATH,
    }
    return df, eval_df, cls_cm, summary


def style_sheet_basic(ws):
    header_fill = PatternFill('solid', fgColor='1F4E78')
    header_font = Font(color='FFFFFF', bold=True)
    center = Alignment(horizontal='center', vertical='center')
    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = center
    ws.freeze_panes = 'A2'
    for col_cells in ws.columns:
        length = 0
        col_letter = get_column_letter(col_cells[0].column)
        for c in col_cells:
            try:
                length = max(length, len(str(c.value)) if c.value is not None else 0)
            except Exception:
                pass
        ws.column_dimensions[col_letter].width = min(max(length + 2, 10), 42)


def write_excel(df: pd.DataFrame, eval_df: pd.DataFrame, cls_cm: np.ndarray, summary: Dict[str, object], out_path: str):
    ensure_dir_for_file(out_path)

    cm_df = pd.DataFrame(
        cls_cm,
        index=[f'GT_{c}' for c in MODEL_CLS_NAMES],
        columns=[f'Pred_{c}' for c in MODEL_CLS_NAMES],
    )
    summary_df = pd.DataFrame({'metric': list(summary.keys()), 'value': list(summary.values())})

    with pd.ExcelWriter(out_path, engine='openpyxl') as writer:
        df.to_excel(writer, sheet_name='PerImage', index=False)
        eval_df.to_excel(writer, sheet_name='EvaluatedOnly', index=False)
        summary_df.to_excel(writer, sheet_name='Summary', index=False)
        cm_df.to_excel(writer, sheet_name='ConfusionMatrix')

    wb = load_workbook(out_path)
    for sheet_name in ['PerImage', 'EvaluatedOnly', 'Summary', 'ConfusionMatrix']:
        ws = wb[sheet_name]
        style_sheet_basic(ws)

    for sheet_name in ['PerImage', 'EvaluatedOnly']:
        ws = wb[sheet_name]
        headers = {cell.value: cell.column for cell in ws[1] if cell.value}
        for col_name, fmt in {
            'timestamp_s': '0.000000',
            'time_rel_s': '0.000',
        }.items():
            if col_name in headers:
                col_letter = get_column_letter(headers[col_name])
                for cell in ws[col_letter][1:]:
                    cell.number_format = fmt

    wb.save(out_path)


df, eval_df, cls_cm, summary = run_eval()
write_excel(df, eval_df, cls_cm, summary, OUTPUT_XLSX)

print('\n=== RESUMEN ===')
for k, v in summary.items():
    print(f'{k}: {v}')

print(f'\nExcel guardado en: {OUTPUT_XLSX}')
